# 02 Pipeline

Run the canonical multi-source Orders ETL: three registered managed sources, visible project transformation, and one governed curated target.

## Tested with FabricOps

The previous baseline was run in Microsoft Fabric with FabricOps v0.2.0 by Voyce on 6 Aug 2026. This redesign has local structural and public-API compatibility validation only; run it in your configured Fabric workspace before treating it as runtime-validated.

# 0. Environment

Load shared configuration, public APIs, and indexed pipeline state.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    check_dq,
    check_freshness,
    check_schema,
    profile_and_register_table,
    profile_dataframe,
    read_lakehouse_table,
    read_pipeline_prep,
    read_warehouse_query,
    write_lakehouse_table,
    write_pipeline_prep,
    widget_select_data_contract,
    widget_view_catalogue,
)

SOURCES = {}
SOURCE_PREPS = {}
SOURCE_DFS = {}
SOURCE_PROFILES = {}
SOURCE_RESULTS = {}
TARGETS = {}
TARGET_DFS = {}
TARGET_PREPS = {}
TARGET_PROFILES = {}
TARGET_RESULTS = {}

## Select registered identities

Select each named managed table in turn. The curated target is selected now because Orders source preparation reads successful watermark progress from that governed target.

### TARGET — Curated Orders

Select the registered `demo.orders` table in the configured downstream Lakehouse (`unified`). Its `table_id` remains the stable hand-off to later Guided Demo steps.

In [ ]:
target_catalogue = widget_view_catalogue(mode="explore", spark_session=spark)

In [ ]:
target_selection = target_catalogue["get_selection"]()
if not target_selection.get("table_id"):
    raise ValueError("Select the registered curated Orders target first.")
if (
    target_selection["store_type"] != "lakehouse"
    or target_selection["layer"] != "unified"
    or target_selection.get("schema_name") != "demo"
    or target_selection["table_name"] != "orders"
):
    raise ValueError("Select the registered unified Lakehouse table demo.orders.")

TARGETS["curated_orders"] = {
    "table_id": target_selection["table_id"],
    "store_type": target_selection["store_type"],
    "target": target_selection["layer"],
    "schema": target_selection.get("schema_name"),
    "table_name": target_selection["table_name"],
}
target = TARGETS["curated_orders"]
CURATED_ORDERS_TABLE_ID = target["table_id"]
print(f"Curated Orders table_id: {CURATED_ORDERS_TABLE_ID}")

# E. Extract

Orders determines whether the pipeline runs. Products and Order History are full-read reference inputs whenever Orders has work.

## SOURCE 1 — Orders

### Select / Configure

Select the registered Source Lakehouse table `demo.orders`. The configured `incremental_watermark` strategy uses `modified_datetime`; preparation determines the runtime mode: `full_dataset`, `incremental_subset`, or `skip`.

In [ ]:
orders_catalogue = widget_view_catalogue(mode="explore", spark_session=spark)

In [ ]:
orders_selection = orders_catalogue["get_selection"]()
if not orders_selection.get("table_id"):
    raise ValueError("Select the registered Orders source.")
if (
    orders_selection["store_type"] != "lakehouse"
    or orders_selection["layer"] != "source"
    or orders_selection.get("schema_name") != "demo"
    or orders_selection["table_name"] != "orders"
):
    raise ValueError("Select the registered source Lakehouse table demo.orders.")

SOURCES["orders"] = {
    "table_id": orders_selection["table_id"],
    "store_type": "lakehouse",
    "target": "source",
    "schema": "demo",
    "table_name": "orders",
    "read_strategy": "incremental_watermark",
    "watermark_column": "modified_datetime",
    "partition_column": None,
}

### Prepare for Read

Preparation records Source Observation and Lineage evidence, then resolves the bounded physical scope from the governed target state.

In [ ]:
source = SOURCES["orders"]
SOURCE_PREPS["orders"] = read_pipeline_prep(
    source_table_id=source["table_id"],
    source_read_strategy=source["read_strategy"],
    target_table_id=CURATED_ORDERS_TABLE_ID,
    source_watermark_column=source["watermark_column"],
    source_partition_column=source["partition_column"],
)
source_prep = SOURCE_PREPS["orders"]
SOURCE_RESULTS["orders"] = [check_schema(table_id=source["table_id"])]
if source_prep["observation"] is not None:
    SOURCE_RESULTS["orders"].append(check_freshness(source_prep["observation"], table_id=source["table_id"]))
if source_prep["changes"] is not None:
    SOURCE_RESULTS["orders"].append(source_prep["changes"])
if not all(result["can_continue"] for result in SOURCE_RESULTS["orders"]):
    raise RuntimeError("An Orders source Guardrail blocked this run.")

PIPELINE_SHOULD_RUN = source_prep["read_mode"] != "skip"
print(f'Orders runtime read mode: {source_prep["read_mode"]}')
print(f'Orders runtime scope: {source_prep["scope"]}')

### Read

The first run reads the full table, an unchanged source skips physical work, and rows appended from `orders_incremental.csv` naturally produce an incremental subset.

In [ ]:
if PIPELINE_SHOULD_RUN:
    SOURCE_DFS["orders"] = read_lakehouse_table(
        source["table_name"],
        target=source["target"],
        schema=source["schema"],
        spark_session=spark,
        processing_scope=source_prep["scope"],
    )
else:
    SOURCE_DFS["orders"] = None
    print("Orders is unchanged. Physical reads, transformation, and publication are skipped.")

### Guard / Profile

A full read may update the canonical source profile. An incremental slice is diagnostic only and never replaces that full-table snapshot.

In [ ]:
if PIPELINE_SHOULD_RUN:
    source_dq = check_dq(SOURCE_DFS["orders"], table_id=source["table_id"])
    SOURCE_RESULTS["orders"].append(source_dq)
    display(source_dq["summary"])
    if not source_dq["can_continue"]:
        raise RuntimeError("An Orders DQ Guardrail blocked this run.")

    if source_prep["read_mode"] == "full_dataset":
        SOURCE_PROFILES["orders"] = profile_and_register_table(
            SOURCE_DFS["orders"], profile_role="source", table=source_prep["source"]
        )
    elif source_prep["read_mode"] == "incremental_subset":
        SOURCE_PROFILES["orders"] = profile_dataframe(SOURCE_DFS["orders"])
    display(SOURCE_PROFILES["orders"])

## SOURCE 2 — Products

### Select / Configure

Select the registered Source Lakehouse table `demo.products`. It is a `full_dataset` lookup source for every Orders run.

In [ ]:
products_catalogue = widget_view_catalogue(mode="explore", spark_session=spark)

In [ ]:
products_selection = products_catalogue["get_selection"]()
if not products_selection.get("table_id"):
    raise ValueError("Select the registered Products source.")
if (
    products_selection["store_type"] != "lakehouse"
    or products_selection["layer"] != "source"
    or products_selection.get("schema_name") != "demo"
    or products_selection["table_name"] != "products"
):
    raise ValueError("Select the registered source Lakehouse table demo.products.")

SOURCES["products"] = {
    "table_id": products_selection["table_id"], "store_type": "lakehouse",
    "target": "source", "schema": "demo", "table_name": "products",
    "read_strategy": "full_dataset", "watermark_column": None, "partition_column": None,
}

### Prepare for Read / Read

In [ ]:
source = SOURCES["products"]
SOURCE_PREPS["products"] = read_pipeline_prep(
    source_table_id=source["table_id"],
    source_read_strategy=source["read_strategy"],
    source_watermark_column=source["watermark_column"],
    source_partition_column=source["partition_column"],
)
source_prep = SOURCE_PREPS["products"]
SOURCE_RESULTS["products"] = [check_schema(table_id=source["table_id"])]
if not all(result["can_continue"] for result in SOURCE_RESULTS["products"]):
    raise RuntimeError("A Products source Guardrail blocked this run.")
print(f'Products runtime read mode: {source_prep["read_mode"]}')

In [ ]:
if PIPELINE_SHOULD_RUN:
    SOURCE_DFS["products"] = read_lakehouse_table(
        source["table_name"], target=source["target"], schema=source["schema"],
        spark_session=spark, processing_scope=source_prep["scope"],
    )
    source_dq = check_dq(SOURCE_DFS["products"], table_id=source["table_id"])
    SOURCE_RESULTS["products"].append(source_dq)
    if not source_dq["can_continue"]:
        raise RuntimeError("A Products DQ Guardrail blocked this run.")
    SOURCE_PROFILES["products"] = profile_and_register_table(
        SOURCE_DFS["products"], profile_role="source", table=source_prep["source"]
    )
else:
    SOURCE_DFS["products"] = None

## SOURCE 3 — Order History

### Select / Configure

Select the registered Product Warehouse table `demo.order_history`. Its `full_dataset` strategy is physically fulfilled by a useful SQL aggregation rather than a full-table reader.

In [ ]:
history_catalogue = widget_view_catalogue(mode="explore", spark_session=spark)

In [ ]:
history_selection = history_catalogue["get_selection"]()
if not history_selection.get("table_id"):
    raise ValueError("Select the registered Order History source.")
if (
    history_selection["store_type"] != "warehouse"
    or history_selection["layer"] != "product"
    or history_selection.get("schema_name") != "demo"
    or history_selection["table_name"] != "order_history"
):
    raise ValueError("Select the registered Product Warehouse table demo.order_history.")

SOURCES["order_history"] = {
    "table_id": history_selection["table_id"], "store_type": "warehouse",
    "target": "product", "schema": "demo", "table_name": "order_history",
    "read_strategy": "full_dataset", "watermark_column": None, "partition_column": None,
}

### Prepare for Read / Read

In [ ]:
source = SOURCES["order_history"]
SOURCE_PREPS["order_history"] = read_pipeline_prep(
    source_table_id=source["table_id"],
    source_read_strategy=source["read_strategy"],
    source_watermark_column=source["watermark_column"],
    source_partition_column=source["partition_column"],
)
source_prep = SOURCE_PREPS["order_history"]
SOURCE_RESULTS["order_history"] = [check_schema(table_id=source["table_id"])]
if not all(result["can_continue"] for result in SOURCE_RESULTS["order_history"]):
    raise RuntimeError("An Order History source Guardrail blocked this run.")
print(f'Order History runtime read mode: {source_prep["read_mode"]}')

In [ ]:
if PIPELINE_SHOULD_RUN:
    SOURCE_DFS["order_history"] = read_warehouse_query(
        """
        SELECT
            customer_id,
            COUNT(*) AS historical_order_count,
            SUM(net_amount) AS historical_net_amount,
            MAX(order_datetime) AS latest_historical_order_datetime
        FROM demo.order_history
        GROUP BY customer_id
        """,
        target=source["target"],
        spark_session=spark,
    )
    # The SQL result is an aggregate, not the complete registered Warehouse table.
    # Profile it diagnostically without replacing the canonical source snapshot.
    SOURCE_PROFILES["order_history"] = profile_dataframe(SOURCE_DFS["order_history"])
else:
    SOURCE_DFS["order_history"] = None

### Strategy and runtime scope

Configured source strategies are `full_dataset`, `incremental_watermark`, or `incremental_partition`. Preparation separately returns runtime modes `full_dataset`, `incremental_subset`, or `skip`; the notebook does not manufacture incremental rows.

# T. Transform

**Business transformation is project-owned.** Enrich current Orders with product attributes and customer history while retaining `modified_datetime` for governed watermark progress.

In [ ]:
if PIPELINE_SHOULD_RUN:
    orders_df = SOURCE_DFS["orders"].alias("orders")
    products_df = SOURCE_DFS["products"].alias("products")
    history_df = SOURCE_DFS["order_history"].alias("history")

    transformed_df = (
        orders_df
        .join(products_df, on="product_id", how="left")
        .join(history_df, on="customer_id", how="left")
        .withColumn(
            "order_net_amount",
            F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2),
        )
        .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
        .select(
            "order_id", "customer_id", "order_datetime", "modified_datetime",
            "product_id", "product_name", "product_category", "quantity", "unit_price",
            "discount", "order_net_amount", "order_status", "shipping_country",
            "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
        )
    )
    display(transformed_df)

# L. Load

## TARGET — Curated Orders

The selected `CURATED_ORDERS_TABLE_ID` identifies the managed `demo.orders` table in the configured downstream Lakehouse.

### Select Data Contract

In [ ]:
CONTRACT_SELECTION = widget_select_data_contract()

### Guard

In [ ]:
if PIPELINE_SHOULD_RUN:
    TARGET_DFS["curated_orders"] = transformed_df
    target_df = TARGET_DFS["curated_orders"]
    target_schema_result = check_schema(table_id=target["table_id"], dataframe=target_df)
    target_dq_result = check_dq(target_df, table_id=target["table_id"])
    TARGET_RESULTS["curated_orders"] = [target_schema_result, target_dq_result]
    display(target_dq_result["summary"])
    if not all(result["can_continue"] for result in TARGET_RESULTS["curated_orders"]):
        raise RuntimeError("A curated Orders target Guardrail blocked publication.")

### Prepare for Write

Target preparation remains authoritative for governed load strategy, writer mode/options, processing scope, target identity, and Lineage context.

In [ ]:
if PIPELINE_SHOULD_RUN:
    TARGET_PREPS["curated_orders"] = write_pipeline_prep(
        target_df,
        target_table_id=target["table_id"],
        source_preps=[
            SOURCE_PREPS["orders"],
            SOURCE_PREPS["products"],
            SOURCE_PREPS["order_history"],
        ],
    )
    target_prep = TARGET_PREPS["curated_orders"]
    prepared_target_df = target_prep["df"].persist()

### Parallel processing / Publish

The prepared DataFrame is the shared input to Spark-distributed publication and diagnostic profiling. FabricOps does not add Python threading: the public writer consumes every governed preparation value, while the successful managed table is re-read for canonical target evidence.

In [ ]:
if PIPELINE_SHOULD_RUN:
    target_slice_profile = profile_dataframe(prepared_target_df)
    TARGET_RESULTS["curated_orders"].append(target_slice_profile)

    TARGET_RESULTS["curated_orders"].append(write_lakehouse_table(
        prepared_target_df,
        target_prep["target"]["table_name"],
        target=target_prep["target"]["layer"],
        schema=target_prep["target"].get("schema_name"),
        mode=target_prep["mode"],
        options=target_prep["options"],
        load_strategy=target_prep["load_strategy"],
        load_strategy_parameters=target_prep["load_strategy_parameters"],
        processing_scope=target_prep["scope"],
    ))
    prepared_target_df.unpersist()

### Evidence

After successful publication, profile the complete curated table. This writes coherent `METADATA_DATA_PROFILED`, eligible `METADATA_DATA_PROFILED_FREQUENCY`, and `METADATA_DATA_CATALOGUE` records; preparation/publication retain `METADATA_DATA_LINEAGE` participation.

In [ ]:
if PIPELINE_SHOULD_RUN:
    curated_orders_df = read_lakehouse_table(
        target_prep["target"]["table_name"],
        target=target_prep["target"]["layer"],
        schema=target_prep["target"].get("schema_name"),
        spark_session=spark,
    )
    TARGET_PROFILES["curated_orders"] = profile_and_register_table(
        curated_orders_df,
        profile_role="target",
        table=target_prep["target"],
        load_strategy=target_prep["load_strategy"],
        load_strategy_parameters=target_prep["load_strategy_parameters"],
    )
    display(TARGET_PROFILES["curated_orders"])
    print(f"Curated Orders table_id: {CURATED_ORDERS_TABLE_ID}")